# DenseNet121 from Scratch — ISIC 2019 Skin Lesion Classification

## Experiment 02

**Architecture:** DenseNet121  
**Training Strategy:** From Scratch  
**Pretrained Weights:** No  
**Data Augmentation:** No  
**Dataset:** ISIC 2019  
**Number of Classes:** 8  
**Model Input Size:** 224 × 224  
**Cached Image Size:** 256 × 256  
**Framework:** PyTorch  
**Hardware:** Apple Silicon  
**Compute Device:** MPS  

---

## Objective

Train DenseNet121 from randomly initialized weights on the ISIC 2019
skin-lesion dataset.

The same permanent lesion-aware train, validation, and test splits used
for the ResNet50-from-scratch experiment will be reused.

The preprocessing pipeline, class-weighting strategy, optimizer,
learning-rate configuration, evaluation metrics, and major training
settings will remain consistent with the ResNet50 scratch experiment.

This allows a fair architecture-level comparison between:

- ResNet50 trained from scratch
- DenseNet121 trained from scratch

No transfer learning or data augmentation is used in this experiment.

In [ ]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path
import random
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import densenet121

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# Environment Information
# ============================================================

print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

print("PyTorch version :", torch.__version__)
print("NumPy version   :", np.__version__)
print("Pandas version  :", pd.__version__)

print("\nLibraries imported successfully.")

In [ ]:
# ============================================================
# Project Paths and Experiment Configuration
# ============================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "dataset"
SPLIT_DIR = DATA_DIR / "splits"

TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"

MODEL_DIR = PROJECT_DIR / "saved_models"
RESULTS_DIR = PROJECT_DIR / "results"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Experiment Constants
# ============================================================

NUM_CLASSES = 8
IMAGE_SIZE = 224
CACHE_SIZE = 256
RANDOM_SEED = 42


# ============================================================
# Display Configuration
# ============================================================

print("=" * 60)
print("PROJECT CONFIGURATION")
print("=" * 60)

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("Split directory   :", SPLIT_DIR)

print("\nTrain CSV exists :", TRAIN_CSV.exists())
print("Val CSV exists   :", VAL_CSV.exists())
print("Test CSV exists  :", TEST_CSV.exists())

print("\nModel directory   :", MODEL_DIR)
print("Results directory :", RESULTS_DIR)

print("\nNumber of classes :", NUM_CLASSES)
print("Model image size  :", IMAGE_SIZE)
print("Cache image size  :", CACHE_SIZE)
print("Random seed       :", RANDOM_SEED)

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


print("=" * 60)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 60)

print("Python random seed :", RANDOM_SEED)
print("NumPy random seed  :", RANDOM_SEED)
print("PyTorch random seed:", RANDOM_SEED)

print("\nRandom seeds configured successfully.")

In [ ]:
# ============================================================
# Device Configuration
# ============================================================

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")

else:
    DEVICE = torch.device("cpu")


print("=" * 60)
print("DEVICE CONFIGURATION")
print("=" * 60)

print("Selected device :", DEVICE)

if DEVICE.type == "mps":
    print("Apple Metal Performance Shaders (MPS) is available.")

elif DEVICE.type == "cuda":
    print("CUDA GPU is available.")
    print("GPU name:", torch.cuda.get_device_name(0))

else:
    print("Using CPU.")

In [ ]:
# ============================================================
# Load Permanent Dataset Splits
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)


# ============================================================
# Display Split Information
# ============================================================

print("=" * 60)
print("DATASET SPLITS")
print("=" * 60)

print(f"Training samples   : {len(train_df):,}")
print(f"Validation samples : {len(val_df):,}")
print(f"Test samples       : {len(test_df):,}")

print(
    f"Total samples      : "
    f"{len(train_df) + len(val_df) + len(test_df):,}"
)


# ============================================================
# Basic Split Verification
# ============================================================

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nFirst training row:")
display(train_df.head(1))

In [ ]:
# ============================================================
# Build Class Information Table
# ============================================================

class_info = (
    train_df[
        [
            "class_index",
            "class_code",
            "class_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("class_index")
    .reset_index(drop=True)
)


# ============================================================
# Verify Class Mapping
# ============================================================

print("=" * 60)
print("CLASS INFORMATION")
print("=" * 60)

for _, row in class_info.iterrows():

    print(
        f"{int(row['class_index'])} "
        f"- {row['class_code']:5s} "
        f"- {row['class_name']}"
    )


print("\nNumber of unique classes:", len(class_info))

In [ ]:
# ============================================================
# Configure Existing 256x256 Image Cache
# ============================================================

CACHE_DIR = DATA_DIR / f"ISIC_2019_{CACHE_SIZE}"


# ============================================================
# Find Cached Images
# ============================================================

cached_images = list(
    CACHE_DIR.glob("*.jpg")
)


# ============================================================
# Expected Number of Images
# ============================================================

expected_images = (
    len(train_df)
    + len(val_df)
    + len(test_df)
)


# ============================================================
# Cache Verification
# ============================================================

print("=" * 60)
print("IMAGE CACHE VERIFICATION")
print("=" * 60)

print("Cache directory :", CACHE_DIR)
print("Cache exists    :", CACHE_DIR.exists())

print(f"\nCached images   : {len(cached_images):,}")
print(f"Expected images : {expected_images:,}")


if (
    CACHE_DIR.exists()
    and len(cached_images) == expected_images
):

    print(
        "\n✓ Existing 256x256 image cache "
        "is complete and ready to use."
    )

else:

    print(
        "\n⚠ Cache verification failed."
    )

    print(
        "Check the cache folder before "
        "continuing with DenseNet121."
    )

In [ ]:
# ============================================================
# Add Cached Image Paths
# ============================================================

def add_cached_path(dataframe):

    dataframe = dataframe.copy()

    dataframe["cached_image_path"] = (
        dataframe["image_path"]
        .apply(
            lambda path: str(
                CACHE_DIR / Path(path).name
            )
        )
    )

    return dataframe


# ============================================================
# Apply to All Dataset Splits
# ============================================================

train_df = add_cached_path(train_df)
val_df = add_cached_path(val_df)
test_df = add_cached_path(test_df)


# ============================================================
# Verification
# ============================================================

print("=" * 60)
print("CACHED IMAGE PATHS")
print("=" * 60)

print("Example training cached path:")
print(train_df.iloc[0]["cached_image_path"])

print(
    "\nExample file exists:",
    Path(
        train_df.iloc[0]["cached_image_path"]
    ).exists()
)

print(
    "\nMissing cached training images:",
    sum(
        not Path(path).exists()
        for path in train_df["cached_image_path"]
    )
)

print(
    "Missing cached validation images:",
    sum(
        not Path(path).exists()
        for path in val_df["cached_image_path"]
    )
)

print(
    "Missing cached test images:",
    sum(
        not Path(path).exists()
        for path in test_df["cached_image_path"]
    )
)

In [ ]:
# ============================================================
# Image Preprocessing — No Data Augmentation
# ============================================================

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]


base_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGE_MEAN,
        std=IMAGE_STD
    )
])


# ============================================================
# Display Preprocessing Pipeline
# ============================================================

print("=" * 60)
print("IMAGE PREPROCESSING PIPELINE")
print("=" * 60)

print("Input cache size :", f"{CACHE_SIZE} x {CACHE_SIZE}")
print("Model input size :", f"{IMAGE_SIZE} x {IMAGE_SIZE}")

print("\nNormalization mean:", IMAGE_MEAN)
print("Normalization std :", IMAGE_STD)

print("\nPreprocessing pipeline:")
print(base_transform)

In [ ]:
# ============================================================
# Custom ISIC Dataset
# ============================================================

class ISICDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(self.dataframe)


    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["cached_image_path"]
        label = int(row["class_index"])

        with Image.open(image_path) as image:

            image = image.convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

        return image, label


# ============================================================
# Confirmation
# ============================================================

print("=" * 60)
print("CUSTOM DATASET")
print("=" * 60)

print("ISICDataset class created successfully.")

In [ ]:
# ============================================================
# Create PyTorch Dataset Objects
# ============================================================

train_dataset = ISICDataset(
    dataframe=train_df,
    transform=base_transform
)

val_dataset = ISICDataset(
    dataframe=val_df,
    transform=base_transform
)

test_dataset = ISICDataset(
    dataframe=test_df,
    transform=base_transform
)


# ============================================================
# Display Dataset Sizes
# ============================================================

print("=" * 60)
print("PYTORCH DATASETS")
print("=" * 60)

print(f"Training samples   : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")
print(f"Test samples       : {len(test_dataset):,}")

print(
    f"Total samples      : "
    f"{len(train_dataset) + len(val_dataset) + len(test_dataset):,}"
)

In [ ]:
# ============================================================
# Verify One Processed Sample
# ============================================================

sample_image, sample_label = train_dataset[0]


print("=" * 60)
print("SAMPLE VERIFICATION")
print("=" * 60)

print("Image tensor shape :", sample_image.shape)
print("Image tensor dtype :", sample_image.dtype)

print("Class index        :", sample_label)

print(
    "Class code         :",
    class_info.iloc[sample_label]["class_code"]
)

print(
    "Class name         :",
    class_info.iloc[sample_label]["class_name"]
)


# ============================================================
# Basic Assertions
# ============================================================

assert sample_image.shape == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert sample_image.dtype == torch.float32

assert 0 <= sample_label < NUM_CLASSES


print("\n✓ Sample verification successful.")

In [ ]:
# ============================================================
# DataLoader Configuration
# ============================================================

BATCH_SIZE = 32
NUM_WORKERS = 0


# ============================================================
# Display Configuration
# ============================================================

print("=" * 60)
print("DATALOADER CONFIGURATION")
print("=" * 60)

print("Batch size  :", BATCH_SIZE)
print("Num workers :", NUM_WORKERS)

print("\nConfiguration ready.")

In [ ]:
# ============================================================
# Create DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


# ============================================================
# Display DataLoader Information
# ============================================================

print("=" * 60)
print("DATALOADERS")
print("=" * 60)

print(f"Training batches   : {len(train_loader):,}")
print(f"Validation batches : {len(val_loader):,}")
print(f"Test batches       : {len(test_loader):,}")

print("\nDataLoaders created successfully.")

In [ ]:
# ============================================================
# Verify One Complete Training Batch
# ============================================================

images, labels = next(
    iter(train_loader)
)


print("=" * 60)
print("BATCH VERIFICATION")
print("=" * 60)

print("Images batch shape :", images.shape)
print("Labels batch shape :", labels.shape)

print("Images dtype       :", images.dtype)
print("Labels dtype       :", labels.dtype)

print("\nFirst 8 labels:")
print(labels[:8])


# ============================================================
# Basic Assertions
# ============================================================

assert images.shape[1:] == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert images.dtype == torch.float32
assert labels.dtype == torch.int64

assert len(images) <= BATCH_SIZE
assert len(labels) == len(images)


print("\n✓ Batch verification successful.")

In [ ]:
# ============================================================
# Create DenseNet121 Model From Scratch
# ============================================================

model = densenet121(
    weights=None
)


# ============================================================
# Display Confirmation
# ============================================================

print("=" * 60)
print("DENSENET121 MODEL")
print("=" * 60)

print("DenseNet121 created successfully.")
print("Pretrained weights: No")
print("Training strategy  : From scratch")

In [ ]:
# ============================================================
# Replace DenseNet121 Final Classifier
# ============================================================

in_features = model.classifier.in_features

model.classifier = nn.Linear(
    in_features=in_features,
    out_features=NUM_CLASSES
)


# ============================================================
# Display Classifier Information
# ============================================================

print("=" * 60)
print("DENSENET121 CLASSIFIER")
print("=" * 60)

print("Input features to classifier :", in_features)
print("Output classes               :", NUM_CLASSES)

print("\nNew classifier:")
print(model.classifier)

In [ ]:
# ============================================================
# Move DenseNet121 to Selected Device
# ============================================================

model = model.to(DEVICE)


# ============================================================
# Display Device Information
# ============================================================

print("=" * 60)
print("MODEL DEVICE")
print("=" * 60)

print("DenseNet121 moved to device:", DEVICE)

In [ ]:
# ============================================================
# Count DenseNet121 Parameters
# ============================================================

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_params = (
    total_params
    - trainable_params
)


# ============================================================
# Display Parameter Counts
# ============================================================

print("=" * 60)
print("DENSENET121 PARAMETER COUNT")
print("=" * 60)

print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Frozen parameters    : "
    f"{frozen_params:,}"
)

In [ ]:
# ============================================================
# Inspect DenseNet121 Architecture
# ============================================================

print("=" * 60)
print("DENSENET121 ARCHITECTURE")
print("=" * 60)

print(model)

In [ ]:
# ============================================================
# Test Forward Pass
# ============================================================

model.eval()

with torch.no_grad():

    test_images = images[:2].to(DEVICE)

    test_outputs = model(test_images)


# ============================================================
# Display Shapes
# ============================================================

print("=" * 60)
print("DENSENET121 FORWARD PASS TEST")
print("=" * 60)

print("Input shape  :", test_images.shape)
print("Output shape :", test_outputs.shape)


# ============================================================
# Basic Assertions
# ============================================================

assert test_images.shape == (
    2,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert test_outputs.shape == (
    2,
    NUM_CLASSES
)

print("\n✓ Forward pass successful.")

In [ ]:
# ============================================================
# Inspect Raw DenseNet121 Output
# ============================================================

single_output = test_outputs[0]

predicted_class_index = torch.argmax(
    single_output
).item()


# ============================================================
# Display Raw Prediction
# ============================================================

print("=" * 60)
print("RAW MODEL OUTPUT")
print("=" * 60)

print("Raw logits:")
print(single_output)

print(
    "\nPredicted class index:",
    predicted_class_index
)

print(
    "Predicted class code :",
    class_info.iloc[
        predicted_class_index
    ]["class_code"]
)

print(
    "Predicted class name :",
    class_info.iloc[
        predicted_class_index
    ]["class_name"]
)

In [ ]:
# ============================================================
# Inspect Raw DenseNet121 Output
# ============================================================

single_output = test_outputs[0]

predicted_class_index = torch.argmax(
    single_output
).item()


# ============================================================
# Display Raw Prediction
# ============================================================

print("=" * 60)
print("RAW MODEL OUTPUT")
print("=" * 60)

print("Raw logits:")
print(single_output)

print(
    "\nPredicted class index:",
    predicted_class_index
)

print(
    "Predicted class code :",
    class_info.iloc[
        predicted_class_index
    ]["class_code"]
)

print(
    "Predicted class name :",
    class_info.iloc[
        predicted_class_index
    ]["class_name"]
)

In [ ]:
# ============================================================
# Calculate Class Weights
# ============================================================

class_counts = (
    train_df["class_index"]
    .value_counts()
    .sort_index()
)

total_training_samples = len(train_df)

class_weights = (
    total_training_samples
    / (
        NUM_CLASSES
        * class_counts.values
    )
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


# ============================================================
# Display Class Distribution and Weights
# ============================================================

print("=" * 60)
print("CLASS WEIGHTS")
print("=" * 60)

print("\nTraining class counts:")
print(class_counts)

print("\nCalculated class weights:")

for index, weight in enumerate(class_weights):

    print(
        f"{index} - "
        f"{class_info.iloc[index]['class_code']:5s} - "
        f"{class_info.iloc[index]['class_name']}: "
        f"{weight:.4f}"
    )


print("\nClass weight tensor:")
print(class_weights_tensor)

In [ ]:
# ============================================================
# Weighted CrossEntropy Loss
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# Display Loss Configuration
# ============================================================

print("=" * 60)
print("LOSS FUNCTION")
print("=" * 60)

print("Loss function : Weighted CrossEntropyLoss")
print("Class weights : Enabled")
print("Device        :", DEVICE)

print("\nCriterion:")
print(criterion)

In [ ]:
# ============================================================
# Optimizer Configuration
# ============================================================

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Display Optimizer Configuration
# ============================================================

print("=" * 60)
print("OPTIMIZER")
print("=" * 60)

print("Optimizer     : AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay :", WEIGHT_DECAY)

print("\nOptimizer created successfully.")

In [ ]:
# ============================================================
# Learning Rate Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ============================================================
# Display Scheduler Configuration
# ============================================================

print("=" * 60)
print("LEARNING RATE SCHEDULER")
print("=" * 60)

print("Scheduler : ReduceLROnPlateau")
print("Mode      : min")
print("Factor    : 0.5")
print("Patience  : 2")

print("\nScheduler created successfully.")

In [ ]:
# ============================================================
# Training Configuration
# ============================================================

NUM_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 5


BEST_MODEL_PATH = (
    MODEL_DIR
    / "densenet121_scratch_best.pth"
)


HISTORY_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_history.csv"
)


# ============================================================
# Display Training Configuration
# ============================================================

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

print("Maximum epochs          :", NUM_EPOCHS)
print("Early stopping patience :", EARLY_STOPPING_PATIENCE)

print("\nBest model path:")
print(BEST_MODEL_PATH)

print("\nTraining history path:")
print(HISTORY_PATH)

In [ ]:
# ============================================================
# Train One Epoch
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    running_correct = 0
    running_samples = 0


    for images, labels in data_loader:

        # ----------------------------------------------------
        # Move current batch to selected device
        # ----------------------------------------------------

        images = images.to(device)
        labels = labels.to(device)


        # ----------------------------------------------------
        # Clear gradients from previous batch
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward Pass
        # ----------------------------------------------------

        outputs = model(images)


        # ----------------------------------------------------
        # Calculate Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Update Model Parameters
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Accumulate Loss
        # ----------------------------------------------------

        batch_size = images.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )


        # ----------------------------------------------------
        # Calculate Predictions
        # ----------------------------------------------------

        predictions = torch.argmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Count Correct Predictions
        # ----------------------------------------------------

        running_correct += (
            predictions == labels
        ).sum().item()

        running_samples += batch_size


    # ========================================================
    # Calculate Epoch Metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / running_samples
    )

    epoch_accuracy = (
        running_correct
        / running_samples
    )


    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# Confirmation
# ============================================================

print("=" * 60)
print("TRAINING FUNCTION")
print("=" * 60)

print("train_one_epoch() created successfully.")

In [ ]:
# ============================================================
# Validate One Epoch
# ============================================================

def validate_one_epoch(
    model,
    data_loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_samples = 0

    all_labels = []
    all_predictions = []


    with torch.no_grad():

        for images, labels in data_loader:

            # ------------------------------------------------
            # Move batch to selected device
            # ------------------------------------------------

            images = images.to(device)
            labels = labels.to(device)


            # ------------------------------------------------
            # Forward Pass
            # ------------------------------------------------

            outputs = model(images)


            # ------------------------------------------------
            # Calculate Validation Loss
            # ------------------------------------------------

            loss = criterion(
                outputs,
                labels
            )


            # ------------------------------------------------
            # Accumulate Loss
            # ------------------------------------------------

            batch_size = images.size(0)

            running_loss += (
                loss.item()
                * batch_size
            )


            # ------------------------------------------------
            # Predicted Classes
            # ------------------------------------------------

            predictions = torch.argmax(
                outputs,
                dim=1
            )


            # ------------------------------------------------
            # Count Correct Predictions
            # ------------------------------------------------

            running_correct += (
                predictions == labels
            ).sum().item()

            running_samples += batch_size


            # ------------------------------------------------
            # Store Labels and Predictions
            # ------------------------------------------------

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    # ========================================================
    # Calculate Epoch Metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / running_samples
    )

    epoch_accuracy = (
        running_correct
        / running_samples
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )


    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )


# ============================================================
# Confirmation
# ============================================================

print("=" * 60)
print("VALIDATION FUNCTION")
print("=" * 60)

print("validate_one_epoch() created successfully.")

In [ ]:
# ============================================================
# Training History and Best-Model Tracking
# ============================================================

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "learning_rate": [],
    "epoch_time_seconds": []
}


# ============================================================
# Best Model Tracking
# ============================================================

best_val_f1 = -1.0

epochs_without_improvement = 0


# ============================================================
# Display Initial State
# ============================================================

print("=" * 60)
print("TRAINING HISTORY INITIALIZATION")
print("=" * 60)

print("Best validation Macro F1     :", best_val_f1)
print(
    "Epochs without improvement  :",
    epochs_without_improvement
)

print("\nHistory fields:")

for key in history.keys():
    print("-", key)

print("\nTraining history initialized successfully.")

In [ ]:
# ============================================================
# One-Batch Forward + Backward Sanity Check
# ============================================================

# Use evaluation mode so BatchNorm running statistics are not
# modified during this sanity check.
# Gradients are still enabled unless torch.no_grad() is used.
model.eval()


# ============================================================
# Get One Training Batch
# ============================================================

sanity_images, sanity_labels = next(
    iter(train_loader)
)

sanity_images = sanity_images.to(DEVICE)
sanity_labels = sanity_labels.to(DEVICE)


# ============================================================
# Clear Existing Gradients
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# Forward Pass
# ============================================================

sanity_outputs = model(
    sanity_images
)


# ============================================================
# Calculate Loss
# ============================================================

sanity_loss = criterion(
    sanity_outputs,
    sanity_labels
)


# ============================================================
# Backward Pass
# ============================================================

sanity_loss.backward()


# ============================================================
# Display Results
# ============================================================

print("=" * 60)
print("ONE-BATCH FORWARD + BACKWARD SANITY CHECK")
print("=" * 60)

print("Input shape  :", sanity_images.shape)
print("Output shape :", sanity_outputs.shape)

print(
    "Loss         :",
    f"{sanity_loss.item():.4f}"
)

print("\nDevice:")
print("Images :", sanity_images.device)
print("Labels :", sanity_labels.device)

print(
    "\n✓ Forward pass successful."
)

print(
    "✓ Loss calculation successful."
)

print(
    "✓ Backpropagation successful."
)


# ============================================================
# Basic Assertions
# ============================================================

assert sanity_outputs.shape == (
    sanity_images.size(0),
    NUM_CLASSES
)

assert torch.isfinite(
    sanity_loss
).item()


# ============================================================
# Clear Sanity-Check Gradients
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)

print(
    "\n✓ Sanity-check gradients cleared."
)

In [ ]:
# ============================================================
# Save DenseNet121 Experiment Configuration
# ============================================================

experiment_config = {
    "model_name": "DenseNet121",
    "training_type": "from_scratch",
    "pretrained_weights": False,
    "data_augmentation": False,

    "num_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "cache_size": CACHE_SIZE,

    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,

    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,

    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "min",
    "scheduler_factor": 0.5,
    "scheduler_patience": 2,

    "num_epochs": NUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,

    "loss_function": "Weighted CrossEntropyLoss",
    "random_seed": RANDOM_SEED,

    "device": str(DEVICE)
}


# ============================================================
# Configuration File Path
# ============================================================

CONFIG_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_config.json"
)


# ============================================================
# Save Configuration
# ============================================================

with open(
    CONFIG_PATH,
    "w"
) as file:

    json.dump(
        experiment_config,
        file,
        indent=4
    )


# ============================================================
# Display Saved Configuration
# ============================================================

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print(
    "Configuration saved to:"
)

print(CONFIG_PATH)

print(
    "\nDenseNet121 scratch experiment "
    "configuration saved successfully."
)

In [ ]:
# ============================================================
# Full DenseNet121 Training Loop
# ============================================================

training_start_time = time.time()


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    # ========================================================
    # Start Epoch Timer
    # ========================================================

    epoch_start_time = time.time()


    # ========================================================
    # Training Phase
    # ========================================================

    train_loss, train_accuracy = train_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )


    # ========================================================
    # Validation Phase
    # ========================================================

    (
        val_loss,
        val_accuracy,
        val_macro_f1
    ) = validate_one_epoch(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )


    # ========================================================
    # Learning Rate Scheduler
    # ========================================================

    scheduler.step(
        val_loss
    )


    # ========================================================
    # Get Current Learning Rate
    # ========================================================

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    # ========================================================
    # Epoch Time
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start_time
    )


    # ========================================================
    # Store Training History
    # ========================================================

    history["epoch"].append(
        epoch
    )

    history["train_loss"].append(
        train_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_accuracy"].append(
        val_accuracy
    )

    history["val_macro_f1"].append(
        val_macro_f1
    )

    history["learning_rate"].append(
        current_lr
    )

    history["epoch_time_seconds"].append(
        epoch_time
    )


    # ========================================================
    # Save History After Every Epoch
    # ========================================================

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        HISTORY_PATH,
        index=False
    )


    # ========================================================
    # Display Epoch Results
    # ========================================================

    print(
        "\n"
        + "=" * 75
    )

    print(
        f"Epoch {epoch}/{NUM_EPOCHS}"
    )

    print(
        "=" * 75
    )

    print(
        f"Train Loss      : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy  : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Loss : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Acc  : "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Validation F1   : "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Learning Rate   : "
        f"{current_lr:.8f}"
    )

    print(
        f"Epoch Time      : "
        f"{epoch_time / 60:.2f} minutes"
    )


    # ========================================================
    # Best Model Checkpoint
    # ========================================================

    if val_macro_f1 > best_val_f1:

        best_val_f1 = (
            val_macro_f1
        )

        epochs_without_improvement = 0


        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )


        print(
            "\n✓ New best DenseNet121 model saved."
        )

        print(
            f"Best Validation Macro F1: "
            f"{best_val_f1:.4f}"
        )


    else:

        epochs_without_improvement += 1


        print(
            "\nNo new best model."
        )

        print(
            "Epochs without improvement:",
            epochs_without_improvement
        )


    # ========================================================
    # Early Stopping
    # ========================================================

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\n"
            + "=" * 75
        )

        print(
            "EARLY STOPPING TRIGGERED"
        )

        print(
            "=" * 75
        )

        print(
            f"No improvement in Validation "
            f"Macro F1 for "
            f"{EARLY_STOPPING_PATIENCE} "
            f"consecutive epochs."
        )

        break


# ============================================================
# Total Training Time
# ============================================================

total_training_time = (
    time.time()
    - training_start_time
)

epochs_completed = len(
    history["epoch"]
)


# ============================================================
# Final Training Summary
# ============================================================

print(
    "\n"
    + "=" * 75
)

print(
    "DENSENET121 TRAINING COMPLETE"
)

print(
    "=" * 75
)

print(
    "Epochs completed          :",
    epochs_completed
)

print(
    f"Best Validation Macro F1 : "
    f"{best_val_f1:.4f}"
)

print(
    f"Total Training Time      : "
    f"{total_training_time / 60:.2f} minutes"
)

print(
    "\nBest model saved to:"
)

print(
    BEST_MODEL_PATH
)

print(
    "\nTraining history saved to:"
)

print(
    HISTORY_PATH
)

In [ ]:
# ============================================================
# Load DenseNet121 Training History
# ============================================================

history_df = pd.read_csv(
    HISTORY_PATH
)


# ============================================================
# Identify Best Epoch
# ============================================================

best_epoch_index = (
    history_df["val_macro_f1"]
    .idxmax()
)

best_epoch_row = (
    history_df
    .loc[best_epoch_index]
)

best_epoch = int(
    best_epoch_row["epoch"]
)


# ============================================================
# Display Training Summary
# ============================================================

print("=" * 70)
print("DENSENET121 TRAINING HISTORY SUMMARY")
print("=" * 70)

print(
    "Epochs completed :",
    len(history_df)
)

print(
    "Best epoch       :",
    best_epoch
)

print(
    f"Best Val Macro F1: "
    f"{best_epoch_row['val_macro_f1']:.4f}"
)

print(
    f"Val Accuracy     : "
    f"{best_epoch_row['val_accuracy'] * 100:.2f}%"
)

print(
    f"Val Loss         : "
    f"{best_epoch_row['val_loss']:.4f}"
)

print(
    f"Train Accuracy   : "
    f"{best_epoch_row['train_accuracy'] * 100:.2f}%"
)

print(
    f"Train Loss       : "
    f"{best_epoch_row['train_loss']:.4f}"
)

print(
    f"Learning Rate    : "
    f"{best_epoch_row['learning_rate']:.8f}"
)

print(
    f"Epoch Time       : "
    f"{best_epoch_row['epoch_time_seconds'] / 60:.2f} minutes"
)


# ============================================================
# Display Complete History
# ============================================================

print("\nComplete training history:")

display(
    history_df
)

In [ ]:
best_model_state = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(best_model_state)
model.eval()


# ============================================================
# Plot Training and Validation Loss
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    marker="o",
    label="Validation Loss"
)


# ============================================================
# Highlight Best Epoch
# ============================================================

plt.axvline(
    x=best_epoch,
    linestyle="--",
    label=f"Best Epoch = {best_epoch}"
)


# ============================================================
# Plot Formatting
# ============================================================

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "DenseNet121 from Scratch - Training vs Validation Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Plot Training Accuracy, Validation Accuracy,
# and Validation Macro F1
# ============================================================

plt.figure(
    figsize=(10, 6)
)


# ============================================================
# Training Accuracy
# ============================================================

plt.plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    marker="o",
    label="Training Accuracy"
)


# ============================================================
# Validation Accuracy
# ============================================================

plt.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    marker="o",
    label="Validation Accuracy"
)


# ============================================================
# Validation Macro F1
# ============================================================

plt.plot(
    history_df["epoch"],
    history_df["val_macro_f1"],
    marker="o",
    label="Validation Macro F1"
)


# ============================================================
# Highlight Best Epoch
# ============================================================

plt.axvline(
    x=best_epoch,
    linestyle="--",
    label=f"Best Epoch = {best_epoch}"
)


# ============================================================
# Plot Formatting
# ============================================================

plt.xlabel("Epoch")
plt.ylabel("Score")

plt.title(
    "DenseNet121 from Scratch - Accuracy and Macro F1"
)

plt.ylim(
    0,
    1
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Load Best DenseNet121 Checkpoint
# ============================================================

print("=" * 60)
print("LOADING BEST DENSENET121 CHECKPOINT")
print("=" * 60)


# ============================================================
# Verify Checkpoint Exists
# ============================================================

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best model checkpoint not found:\n"
        f"{BEST_MODEL_PATH}"
    )


# ============================================================
# Load Saved State Dictionary
# ============================================================

best_model_state = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)


# ============================================================
# Restore Model Weights
# ============================================================

model.load_state_dict(
    best_model_state
)

model = model.to(DEVICE)


# ============================================================
# Switch to Evaluation Mode
# ============================================================

model.eval()


# ============================================================
# Confirmation
# ============================================================

print("Checkpoint loaded successfully.")

print(
    "\nBest checkpoint:"
)
print(BEST_MODEL_PATH)

print(
    "\nBest validation epoch:",
    best_epoch
)

print(
    f"Best validation Macro F1: "
    f"{best_epoch_row['val_macro_f1']:.4f}"
)

print(
    "\nModel device:",
    next(model.parameters()).device
)

print(
    "\nModel mode:",
    "Evaluation"
)

print(
    "\n✓ Best DenseNet121 model is ready for final test evaluation."
)

In [ ]:
# ============================================================
# Evaluate Best DenseNet121 on Test Set
# ============================================================

model.eval()

test_running_loss = 0.0
test_running_samples = 0

all_test_labels = []
all_test_predictions = []
all_test_probabilities = []


# ============================================================
# Test Loop
# ============================================================

with torch.no_grad():

    for images, labels in test_loader:

        # ----------------------------------------------------
        # Move Batch to Device
        # ----------------------------------------------------

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)


        # ----------------------------------------------------
        # Forward Pass
        # ----------------------------------------------------

        outputs = model(images)


        # ----------------------------------------------------
        # Calculate Test Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Convert Logits to Probabilities
        # ----------------------------------------------------

        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Predicted Classes
        # ----------------------------------------------------

        predictions = torch.argmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Accumulate Loss
        # ----------------------------------------------------

        batch_size = images.size(0)

        test_running_loss += (
            loss.item()
            * batch_size
        )

        test_running_samples += batch_size


        # ----------------------------------------------------
        # Store Results on CPU
        # ----------------------------------------------------

        all_test_labels.extend(
            labels.cpu().numpy()
        )

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )

        all_test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# ============================================================
# Convert Stored Results to NumPy Arrays
# ============================================================

all_test_labels = np.array(
    all_test_labels
)

all_test_predictions = np.array(
    all_test_predictions
)

all_test_probabilities = np.array(
    all_test_probabilities
)


# ============================================================
# Calculate Final Test Metrics
# ============================================================

test_loss = (
    test_running_loss
    / test_running_samples
)

test_accuracy = accuracy_score(
    all_test_labels,
    all_test_predictions
)

test_macro_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)

test_weighted_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="weighted"
)


# ============================================================
# Display Final Test Results
# ============================================================

print("=" * 70)
print("FINAL DENSENET121 TEST RESULTS")
print("=" * 70)

print(
    f"Test Loss        : "
    f"{test_loss:.4f}"
)

print(
    f"Test Accuracy    : "
    f"{test_accuracy:.4f}"
)

print(
    f"Test Accuracy    : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Test Macro F1    : "
    f"{test_macro_f1:.4f}"
)

print(
    f"Test Weighted F1 : "
    f"{test_weighted_f1:.4f}"
)

print(
    "\nTest samples evaluated:",
    test_running_samples
)


# ============================================================
# Basic Verification
# ============================================================

assert (
    len(all_test_labels)
    == len(test_dataset)
)

assert (
    len(all_test_predictions)
    == len(test_dataset)
)

assert all_test_probabilities.shape == (
    len(test_dataset),
    NUM_CLASSES
)

print(
    "\n✓ Full test-set evaluation completed successfully."
)

In [ ]:
# ============================================================
# Detailed Classification Report
# ============================================================

class_names = (
    class_info
    .sort_values("class_index")["class_name"]
    .tolist()
)


classification_report_text = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=class_names,
    digits=4
)


print("=" * 80)
print("DENSENET121 CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report_text
)

In [ ]:
# ============================================================
# Convert Classification Report to DataFrame
# ============================================================

classification_report_dict = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=class_names,
    output_dict=True
)


report_df = pd.DataFrame(
    classification_report_dict
).transpose()


# ============================================================
# Display Per-Class Results
# ============================================================

print("=" * 80)
print("DENSENET121 PER-CLASS RESULTS")
print("=" * 80)

display(
    report_df
)

In [ ]:
# ============================================================
# Save DenseNet121 Test Results
# ============================================================

TEST_RESULTS_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_test_results.json"
)

PER_CLASS_RESULTS_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_per_class_results.csv"
)


# ============================================================
# Overall Test Results
# ============================================================

test_results = {
    "model_name": "DenseNet121",
    "training_type": "from_scratch",

    "best_epoch": int(best_epoch),

    "best_validation_macro_f1": float(
        best_epoch_row["val_macro_f1"]
    ),

    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),

    "test_samples": int(
        len(all_test_labels)
    )
}


# ============================================================
# Save Overall Metrics to JSON
# ============================================================

with open(
    TEST_RESULTS_PATH,
    "w"
) as file:

    json.dump(
        test_results,
        file,
        indent=4
    )


# ============================================================
# Save Per-Class Metrics to CSV
# ============================================================

report_df.to_csv(
    PER_CLASS_RESULTS_PATH
)


# ============================================================
# Display Saved Results
# ============================================================

print("=" * 70)
print("DENSENET121 RESULTS SAVED")
print("=" * 70)

print("\nOverall test results:")
print(TEST_RESULTS_PATH)

print("\nPer-class results:")
print(PER_CLASS_RESULTS_PATH)

print("\nSaved test metrics:")

for key, value in test_results.items():
    print(
        f"{key}: {value}"
    )

print(
    "\n✓ DenseNet121 evaluation results "
    "saved successfully."
)

In [ ]:
# ============================================================
# Raw Confusion Matrix
# ============================================================

confusion_matrix_raw = confusion_matrix(
    all_test_labels,
    all_test_predictions
)


# ============================================================
# Convert to DataFrame
# ============================================================

confusion_matrix_df = pd.DataFrame(
    confusion_matrix_raw,
    index=class_names,
    columns=class_names
)


# ============================================================
# Save Confusion Matrix
# ============================================================

CONFUSION_MATRIX_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_confusion_matrix.csv"
)

confusion_matrix_df.to_csv(
    CONFUSION_MATRIX_PATH
)


# ============================================================
# Display Matrix
# ============================================================

print("=" * 80)
print("DENSENET121 RAW CONFUSION MATRIX")
print("=" * 80)

display(
    confusion_matrix_df
)

print(
    "\nConfusion matrix saved to:"
)

print(
    CONFUSION_MATRIX_PATH
)

print(
    "\n✓ Raw confusion matrix created and saved successfully."
)

In [ ]:
# ============================================================
# Plot Raw Confusion Matrix
# ============================================================

plt.figure(
    figsize=(12, 10)
)

plt.imshow(
    confusion_matrix_raw
)

plt.title(
    "DenseNet121 from Scratch - Raw Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)


# ============================================================
# Class Labels
# ============================================================

plt.xticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names
)


# ============================================================
# Add Values Inside Each Cell
# ============================================================

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            confusion_matrix_raw[i, j],
            ha="center",
            va="center"
        )


# ============================================================
# Add Color Scale
# ============================================================

plt.colorbar(
    label="Number of Samples"
)


# ============================================================
# Final Layout
# ============================================================

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Normalized Confusion Matrix
# ============================================================

confusion_matrix_normalized = (
    confusion_matrix_raw
    / confusion_matrix_raw.sum(
        axis=1,
        keepdims=True
    )
)


# ============================================================
# Convert to DataFrame
# ============================================================

confusion_matrix_normalized_df = pd.DataFrame(
    confusion_matrix_normalized,
    index=class_names,
    columns=class_names
)


# ============================================================
# Save Normalized Confusion Matrix
# ============================================================

NORMALIZED_CONFUSION_MATRIX_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_confusion_matrix_normalized.csv"
)

confusion_matrix_normalized_df.to_csv(
    NORMALIZED_CONFUSION_MATRIX_PATH
)


# ============================================================
# Display Matrix
# ============================================================

print("=" * 80)
print("DENSENET121 NORMALIZED CONFUSION MATRIX")
print("=" * 80)

display(
    confusion_matrix_normalized_df.round(4)
)

print(
    "\nNormalized confusion matrix saved to:"
)

print(
    NORMALIZED_CONFUSION_MATRIX_PATH
)

print(
    "\n✓ Normalized confusion matrix created successfully."
)

In [ ]:
# ============================================================
# Plot Normalized Confusion Matrix
# ============================================================

plt.figure(
    figsize=(12, 10)
)

plt.imshow(
    confusion_matrix_normalized,
    vmin=0,
    vmax=1
)

plt.title(
    "DenseNet121 from Scratch - Normalized Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)


# ============================================================
# Class Labels
# ============================================================

plt.xticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names
)


# ============================================================
# Add Percentages Inside Each Cell
# ============================================================

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        value = (
            confusion_matrix_normalized[i, j]
            * 100
        )

        plt.text(
            j,
            i,
            f"{value:.1f}%",
            ha="center",
            va="center"
        )


# ============================================================
# Add Color Scale
# ============================================================

plt.colorbar(
    label="Proportion"
)


# ============================================================
# Final Layout
# ============================================================

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Create Test Prediction Table
# ============================================================

predicted_confidence = (
    all_test_probabilities.max(
        axis=1
    )
)

predicted_class_indices = (
    all_test_predictions
)

actual_class_indices = (
    all_test_labels
)


# ============================================================
# Convert Class Indices to Names
# ============================================================

actual_class_names = [
    class_names[index]
    for index in actual_class_indices
]

predicted_class_names = [
    class_names[index]
    for index in predicted_class_indices
]


# ============================================================
# Build Prediction DataFrame
# ============================================================

test_predictions_df = pd.DataFrame({
    "actual_class_index": actual_class_indices,
    "actual_class_name": actual_class_names,

    "predicted_class_index": predicted_class_indices,
    "predicted_class_name": predicted_class_names,

    "prediction_confidence": predicted_confidence
})


# ============================================================
# Add Correct / Incorrect Column
# ============================================================

test_predictions_df["is_correct"] = (
    test_predictions_df["actual_class_index"]
    ==
    test_predictions_df["predicted_class_index"]
)


# ============================================================
# Add Probability for Each Class
# ============================================================

for class_index, class_name in enumerate(
    class_names
):

    probability_column_name = (
        f"probability_{class_index}_"
        + class_name
        .lower()
        .replace(" ", "_")
    )

    test_predictions_df[
        probability_column_name
    ] = all_test_probabilities[
        :,
        class_index
    ]


# ============================================================
# Save Predictions
# ============================================================

TEST_PREDICTIONS_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_test_predictions.csv"
)

test_predictions_df.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False
)


# ============================================================
# Display Preview
# ============================================================

print("=" * 80)
print("DENSENET121 TEST PREDICTIONS")
print("=" * 80)

print(
    "Total predictions:",
    len(test_predictions_df)
)

print(
    "Correct predictions:",
    test_predictions_df["is_correct"].sum()
)

print(
    "Incorrect predictions:",
    (~test_predictions_df["is_correct"]).sum()
)

print("\nSaved to:")
print(TEST_PREDICTIONS_PATH)

print("\nPrediction preview:")

display(
    test_predictions_df.head(10)
)

print(
    "\n✓ Test predictions saved successfully."
)

In [ ]:
# ============================================================
# Attach Image Information to Test Predictions
# ============================================================

test_metadata = (
    test_df
    .reset_index(drop=True)
)

test_predictions_with_paths_df = (
    test_predictions_df.copy()
)

test_predictions_with_paths_df[
    "image_name"
] = test_metadata["image_path"].apply(
    lambda path: Path(path).name
)

test_predictions_with_paths_df[
    "image_path"
] = test_metadata["image_path"]

test_predictions_with_paths_df[
    "cached_image_path"
] = test_metadata["cached_image_path"]


# ============================================================
# Extract Misclassified Predictions
# ============================================================

misclassified_df = (
    test_predictions_with_paths_df[
        test_predictions_with_paths_df["is_correct"] == False
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# Sort by Prediction Confidence
# ============================================================

misclassified_df = (
    misclassified_df
    .sort_values(
        by="prediction_confidence",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# Save Misclassified Predictions
# ============================================================

MISCLASSIFIED_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_misclassified_predictions.csv"
)

misclassified_df.to_csv(
    MISCLASSIFIED_PATH,
    index=False
)


# ============================================================
# Display Summary
# ============================================================

total_test_samples = len(
    test_predictions_with_paths_df
)

total_misclassified = len(
    misclassified_df
)

total_correct = (
    total_test_samples
    - total_misclassified
)


print("=" * 80)
print("DENSENET121 MISCLASSIFIED TEST SAMPLES")
print("=" * 80)

print(
    "Total test samples      :",
    total_test_samples
)

print(
    "Correct predictions     :",
    total_correct
)

print(
    "Misclassified samples   :",
    total_misclassified
)

print(
    "Misclassification rate  :",
    f"{(total_misclassified / total_test_samples) * 100:.2f}%"
)


print("\nSaved to:")
print(
    MISCLASSIFIED_PATH
)


# ============================================================
# Display Highest-Confidence Errors
# ============================================================

print(
    "\nHighest-confidence misclassifications:"
)

display(
    misclassified_df[
        [
            "image_name",
            "actual_class_name",
            "predicted_class_name",
            "prediction_confidence"
        ]
    ].head(20)
)

print(
    "\n✓ Misclassified predictions extracted "
    "and saved successfully."
)

In [ ]:
# ============================================================
# Visualize Random Misclassified Test Images
# ============================================================

NUM_IMAGES_TO_SHOW = 8


# ============================================================
# Randomly Sample Misclassified Images
# ============================================================

sample_count = min(
    NUM_IMAGES_TO_SHOW,
    len(misclassified_df)
)

sampled_errors = (
    misclassified_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)


# ============================================================
# Create Figure
# ============================================================

plt.figure(
    figsize=(16, 10)
)


# ============================================================
# Display Images
# ============================================================

for index, row in sampled_errors.iterrows():

    image_path = row[
        "cached_image_path"
    ]

    actual_class = row[
        "actual_class_name"
    ]

    predicted_class = row[
        "predicted_class_name"
    ]

    confidence = row[
        "prediction_confidence"
    ]


    # --------------------------------------------------------
    # Open Image
    # --------------------------------------------------------

    with Image.open(image_path) as image:

        image = image.convert("RGB")


        # ----------------------------------------------------
        # Plot Image
        # ----------------------------------------------------

        plt.subplot(
            2,
            4,
            index + 1
        )

        plt.imshow(
            image
        )

        plt.axis(
            "off"
        )


        # ----------------------------------------------------
        # Image Title
        # ----------------------------------------------------

        plt.title(
            f"Actual: {actual_class}\n"
            f"Predicted: {predicted_class}\n"
            f"Confidence: {confidence * 100:.1f}%",
            fontsize=9
        )


# ============================================================
# Final Layout
# ============================================================

plt.suptitle(
    "DenseNet121 from Scratch - Random Misclassified Test Images",
    fontsize=14
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Visualize Highest-Confidence Misclassifications
# ============================================================

NUM_IMAGES_TO_SHOW = 8


# ============================================================
# Select Highest-Confidence Errors
# ============================================================

top_confidence_errors = (
    misclassified_df
    .sort_values(
        by="prediction_confidence",
        ascending=False
    )
    .head(NUM_IMAGES_TO_SHOW)
    .reset_index(drop=True)
)


# ============================================================
# Create Figure
# ============================================================

plt.figure(
    figsize=(16, 10)
)


# ============================================================
# Display Images
# ============================================================

for index, row in top_confidence_errors.iterrows():

    image_path = row[
        "cached_image_path"
    ]

    actual_class = row[
        "actual_class_name"
    ]

    predicted_class = row[
        "predicted_class_name"
    ]

    confidence = row[
        "prediction_confidence"
    ]


    # --------------------------------------------------------
    # Open Image
    # --------------------------------------------------------

    with Image.open(image_path) as image:

        image = image.convert("RGB")


        # ----------------------------------------------------
        # Plot Image
        # ----------------------------------------------------

        plt.subplot(
            2,
            4,
            index + 1
        )

        plt.imshow(
            image
        )

        plt.axis(
            "off"
        )


        # ----------------------------------------------------
        # Image Title
        # ----------------------------------------------------

        plt.title(
            f"Actual: {actual_class}\n"
            f"Predicted: {predicted_class}\n"
            f"Confidence: {confidence * 100:.1f}%",
            fontsize=9
        )


# ============================================================
# Final Layout
# ============================================================

plt.suptitle(
    "DenseNet121 from Scratch - Highest-Confidence Errors",
    fontsize=14
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Final DenseNet121 Experiment Summary
# ============================================================

final_summary = {
    "Model": "DenseNet121",
    "Training Strategy": "From Scratch",
    "Pretrained Weights": "No",
    "Data Augmentation": "No",

    "Best Epoch": int(
        best_epoch
    ),

    "Best Validation Macro F1": round(
        float(
            best_epoch_row[
                "val_macro_f1"
            ]
        ),
        4
    ),

    "Test Loss": round(
        float(
            test_loss
        ),
        4
    ),

    "Test Accuracy (%)": round(
        float(
            test_accuracy
            * 100
        ),
        2
    ),

    "Test Macro F1": round(
        float(
            test_macro_f1
        ),
        4
    ),

    "Test Weighted F1": round(
        float(
            test_weighted_f1
        ),
        4
    ),

    "Test Samples": int(
        len(
            all_test_labels
        )
    )
}


# ============================================================
# Convert Summary to DataFrame
# ============================================================

final_summary_df = pd.DataFrame(
    list(
        final_summary.items()
    ),
    columns=[
        "Metric",
        "Value"
    ]
)


# ============================================================
# Display Summary
# ============================================================

print("=" * 80)
print("FINAL DENSENET121 EXPERIMENT SUMMARY")
print("=" * 80)

display(
    final_summary_df
)


# ============================================================
# Save Summary
# ============================================================

FINAL_SUMMARY_PATH = (
    RESULTS_DIR
    / "densenet121_scratch_final_summary.csv"
)

final_summary_df.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)


print("\nFinal summary saved to:")
print(
    FINAL_SUMMARY_PATH
)

print(
    "\n✓ DenseNet121 scratch experiment "
    "completed successfully."
)